In [ ]:
%pip install -qU langchain-community pymupdf
!pip install -qU langchain-huggingface sentence-transformers
!pip install -qU langchain-groq
!pip install faiss-cpu

# 1. Loading the document

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

file_path = "/content/the-quran-with-annotated-interpretation-in-modern-english-ali-unal.pdf"
loader = PyMuPDFLoader(file_path)

In [ ]:
docs = loader.load()
# skiping empty pages
non_empty_docs = [d for d in docs if d.page_content.strip()]

# 2. Spliting document into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10000, # 10000 charecters long text
    chunk_overlap=200, # 200 charecters long overlapping
)

split_docs = text_splitter.split_documents(non_empty_docs)

# 3. Embeddings Model

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
texts = [doc.page_content for doc in split_docs] # convert documents into list[str]

In [ ]:
split_docs_embeddings = embed_model.embed_documents(texts) # generate embeddings (list[list[float]])

# 4. FAISS (Facebook AI Similarity Search) vector database

In [ ]:
from langchain_community.vectorstores import FAISS

faiss_db = FAISS.from_documents(
    documents=split_docs,
    embedding=embed_model,
)

# 5. LLM. GROQ (llama-3.3-70b-versatile)

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="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"
)

# 6. Building Prompts and chains

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def ask(query):
    top_docs = faiss_db.similarity_search(query, k=10)
    prompt_template = PromptTemplate.from_template(
      "Give answer according to the following passages in the quran {context}"
      "If the answer is not present in the given context then give answer according to the internet sources."
      "But do inform that there are no passages in the quran about the question."
      "Answer the following question {question}."
    )
    chain = prompt_template | llm | StrOutputParser()
    result = chain.invoke({"context": top_docs, "question": query})
    return result

Relevant questions

In [ ]:
asnwer = ask("What does the Quran say about Day of Judgment?")
print(asnwer)

In [ ]:
asnwer = ask("What of someone donot FAST in the month of RAMADAN?")
print(asnwer)

Irrelevant questions

In [ ]:
asnwer = ask("When did dinosaurs came into being?")
print(asnwer)